<a href="https://colab.research.google.com/github/Dharshini11042004/Adaptive-Threat-Detection-in-Smart-Healthcare-Infrastructure-with-Agentic-AI-using-RL/blob/main/Agent4_Healthcare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# AGENT 4 : HEALTHCARE CONTEXT & SAFETY AGENT
# Autoencoder + Isolation Forest + PPO RL
!pip -q install stable-baselines3 gymnasium
import os
import zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,confusion_matrix
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
# 1. LOAD DATASET
zip_path="/content/preprocessed_Datasets.zip"
extract_path="/content"
with zipfile.ZipFile(zip_path,'r') as z:
    z.extractall(extract_path)
health_path=os.path.join(extract_path,"preprocessed_Datasets","Healthcare")
train_file=os.path.join(health_path,"ICU_dataset_train.csv")
test_file=os.path.join(health_path,"ICU_dataset_test.csv")
train=pd.read_csv(train_file)
test=pd.read_csv(test_file)
print("Train shape:",train.shape)
print("Test shape:",test.shape)
# 2. DATA CLEANING
for col in train.columns:
    if col not in ["label","class"]:
        train[col]=pd.to_numeric(train[col],errors="coerce")
        test[col]=pd.to_numeric(test[col],errors="coerce")
train=train.replace([np.inf,-np.inf],np.nan).fillna(0)
test=test.replace([np.inf,-np.inf],np.nan).fillna(0)
NORMAL_LABEL=train["label"].min()
y_train=np.where(train["label"]==NORMAL_LABEL,0,1)
y_test=np.where(test["label"]==NORMAL_LABEL,0,1)
train=train.drop(columns=["class"])
test=test.drop(columns=["class"])
X_train=train.drop(columns=["label"]).values
X_test=test.drop(columns=["label"]).values
rint("Feature count:",X_train.shape[1])
# 3. NORMALIZATION
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)
# 4. AUTOENCODER
class Autoencoder(nn.Module):
    def __init__(self,dim):
        super().__init__()
        self.encoder=nn.Sequential(
            nn.Linear(dim,64),
            nn.ReLU(),
            nn.Linear(64,32),
            nn.ReLU(),
            nn.Linear(32,16)
        )
        self.decoder=nn.Sequential(
            nn.Linear(16,32),
            nn.ReLU(),
            nn.Linear(32,64),
            nn.ReLU(),
            nn.Linear(64,dim)
        )
    def forward(self,x):
        return self.decoder(self.encoder(x))
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
ae=Autoencoder(X_train.shape[1]).to(device)
optimizer=optim.Adam(ae.parameters(),lr=0.001)
loss_fn=nn.MSELoss()
X_tensor=torch.tensor(X_train,dtype=torch.float32).to(device)
print("\nTraining Autoencoder")
for epoch in range(20):
    recon=ae(X_tensor)
    loss=loss_fn(recon,X_tensor)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if epoch%5==0:
        print("Epoch",epoch,"Loss",loss.item())
# 5. ANOMALY SCORES
with torch.no_grad():
    train_recon=ae(X_tensor).cpu().numpy()
train_mse=np.mean((X_train-train_recon)**2,axis=1)
with torch.no_grad():
    test_recon=ae(torch.tensor(X_test,dtype=torch.float32).to(device)).cpu().numpy()
test_mse=np.mean((X_test-test_recon)**2,axis=1)
# 6. ISOLATION FOREST
print("\nTraining Isolation Forest")
iso=IsolationForest(n_estimators=300,contamination=0.1,random_state=42)
iso.fit(X_train)
train_if=-iso.score_samples(X_train)
test_if=-iso.score_samples(X_test)
# 7. FEATURE FUSION
score_scaler=StandardScaler()
train_scores=score_scaler.fit_transform(np.vstack([train_mse,train_if]).T)
test_scores=score_scaler.transform(np.vstack([test_mse,test_if]).T)
train_state=np.hstack([X_train,train_scores])
test_state=np.hstack([X_test,test_scores])
print("RL State size:",train_state.shape[1])
# 8. RL ENVIRONMENT
class HealthEnv(gym.Env):
    def __init__(self,data,labels):
        super().__init__()
        self.data=data
        self.labels=labels
        self.index=0
        self.actions=[]
        self.rewards=[]
        self.observation_space=spaces.Box(low=-5,high=5,shape=(data.shape[1],),dtype=np.float32)
        self.action_space=spaces.Discrete(3)
    def reset(self,seed=None,options=None):
        self.index=0
        return self.data[self.index],{}
    def step(self,action):
        label=self.labels[self.index]
        if label==1:
            if action==2:
                reward=5
            elif action==1:
                reward=1
            else:
                reward=-5
        else:
            if action==0:
                reward=2
            elif action==1:
                reward=0
            else:
                reward=-2
        self.actions.append(action)
        self.rewards.append(reward)
        self.index+=1
        done=self.index>=len(self.data)
        next_state=np.zeros(self.data.shape[1]) if done else self.data[self.index]
        return next_state,reward,done,False,{}
# 9. TRAIN PPO AGENT
env=HealthEnv(train_state,y_train)
model=PPO(
    "MlpPolicy",
    env,
    learning_rate=0.0003,
    gamma=0.99,
    verbose=1
)
print("\nTraining PPO Agent")
model.learn(total_timesteps=50000)
print("Training complete")
# OUTPUT GENERATION
def generate_outputs(data, mse_scores, if_scores, dataset_name):
    outputs=[]
    for i in range(len(data)):
        state=data[i].reshape(1,-1)
        action,_=model.predict(state)
        anomaly_score=float((mse_scores[i] + if_scores[i]) / 2)
        risk_score=float(anomaly_score * 100)
        if action==0:
            behaviour="Normal"
            decision="Approve Access"
        elif action==1:
            behaviour="Suspicious"
            decision="Delay Response"
        else:
            behaviour="Critical"
            decision="Downgrade Priority"
        outputs.append({
            "Dataset": dataset_name,
            "Index": i,
            "Anomaly Score": anomaly_score,
            "Risk Score": risk_score,
            "Behaviour": behaviour,
            "Action": int(action.item()),
            "Decision": decision
        })
    return outputs
# Generate outputs
train_outputs = generate_outputs(train_state, train_mse, train_if, "Train")
test_outputs  = generate_outputs(test_state, test_mse, test_if, "Test")
all_outputs = train_outputs + test_outputs
# SAVE CSV
output_df = pd.DataFrame(all_outputs)
output_path = "/content/drive/MyDrive/MultiAgentModels/agent_4_outputs.csv"
output_df.to_csv(output_path, index=False)
print("\nSaved → agent_4_outputs.csv")
# EXISTING TEST EVALUATION (UNCHANGED)
env_test=HealthEnv(test_state,y_test)
obs,_=env_test.reset()
done=False
preds=[]
actions=[]
while not done:
    action,_=model.predict(obs)
    actions.append(action)
    pred=0 if action==0 else 1
    preds.append(pred)
    obs,_,done,_,_=env_test.step(action)
preds=np.array(preds)
print("\nAction Distribution")
names=["Approve","Delay","Downgrade"]
unique,counts=np.unique(actions,return_counts=True)
for u,c in zip(unique,counts):
    print(names[u],":",c)
print("\nMODEL PERFORMANCE")
print("Accuracy:",accuracy_score(y_test,preds))
print("Precision:",precision_score(y_test,preds))
print("Recall:",recall_score(y_test,preds))
print("F1 Score:",f1_score(y_test,preds))
print("\nConfusion Matrix")
print(confusion_matrix(y_test,preds))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 5.4 MB/s eta 0:00:00


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Train shape: (132085, 10)
Test shape: (56609, 10)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Feature count: 8


...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

RL State size: 10
Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.

Training PPO Agent


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


-----------------------------
| time/              |      |
|    fps             | 1392 |
|    iterations      | 1    |
|    time_elapsed    | 1    |
|    total_timesteps | 2048 |
-----------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 1033        |
|    iterations           | 2           |
|    time_elapsed         | 3           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.013138859 |
|    clip_fraction        | 0.142       |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.09       |
|    explained_variance   | -0.0025     |
|    learning_rate        | 0.0003      |
|    loss                 | 52.1        |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.0221     |
|    value_loss           | 76.7        |
-----------------------------------------
----------------------------------

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
